In [1]:
import os
import sys

# Task 1 - VQA notebook

This notebook runs Task 1 as a direct Visual Question Answering call: one image + one question + one answer.

In [3]:
sys.path.insert(0, os.path.abspath('promptomatix/src'))

from promptomatix.vqa import answer_visual_question

# Gemini example:
os.environ['VQA_MODEL_PROVIDER'] = 'gemini'
os.environ['VQA_MODEL'] = 'gemini-2.5-flash'

provider = os.getenv('VQA_MODEL_PROVIDER', 'gemini')
model = os.getenv('VQA_MODEL', 'gemini-2.5-flash')
optimizer_model = model if '/' in model else f'gemini/{model}'
api_base = os.getenv('VQA_API_BASE') or os.getenv('OPTIMIZER_API_BASE')

api_key = (
    os.getenv('VQA_API_KEY')
    or os.getenv('GEMINI_API_KEY')
    or os.getenv('GOOGLE_API_KEY')
    or os.getenv('OPENAI_API_KEY')
)
serpapi_api_key = os.getenv('SERPAPI_API_KEY')


if not api_key:
    raise RuntimeError('Set GEMINI_API_KEY, GOOGLE_API_KEY, VQA_API_KEY, or OPENAI_API_KEY before running this notebook.')

print('Provider:', provider)
print('Model:', model)
print('Optimizer model:', optimizer_model)
print('SerpApi key present:', bool(serpapi_api_key))
print('API base:', api_base or '(provider default)')

16:28:33 - LiteLLM:WARNING: common_utils.py:979 - litellm: could not pre-load bedrock-runtime response stream shape — Bedrock event-stream decoding will be unavailable. Error: No module named 'botocore'
16:28:33 - LiteLLM:WARNING: common_utils.py:24 - litellm: could not pre-load sagemaker-runtime response stream shape — SageMaker event-stream decoding will be unavailable. Error: No module named 'botocore'
/opt/anaconda3/envs/mnlp_exercises/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Provider: gemini
Model: gemini-2.5-flash
Optimizer model: gemini/gemini-2.5-flash
SerpApi key present: True
API base: (provider default)


In [4]:
image = 'https://media.audubon.org/nas_birdapi_hero/h_a1_7443_5_painted-bunting_julie_torkomian_adult-male.jpg'
question = 'What kind of bird is in this picture?'

print('Image:', image)
print('Question:', question)

Image: https://media.audubon.org/nas_birdapi_hero/h_a1_7443_5_painted-bunting_julie_torkomian_adult-male.jpg
Question: What kind of bird is in this picture?


In [5]:
answer = answer_visual_question(
    image=image,
    prompt=question,
    model_provider=provider,
    model_name=model,
    model_api_key=api_key,
    model_api_base=api_base,
    max_tokens=512,
)

print('Answer:', answer)

Answer: Based on the vibrant and distinct coloration visible in the image, with a blue head, green back, and red underparts, the bird is a Painted Bunting.


## image_pool -> synthetic VQA data

This section exercises the new multimodal synthetic generation flow added for Part 1.2.
It builds a VQA config with an explicit `image_pool`, then asks `PromptOptimizer` to synthesize grounded question/answer pairs from those images.

In [6]:
from io import BytesIO
from pathlib import Path
from pprint import pprint
from urllib.parse import urlparse

import requests
from PIL import Image

from promptomatix.core.config import Config
from promptomatix.core.optimizer import PromptOptimizer

cache_dir = Path('promptomatix/examples/outputs/fetched_bird_pool')
cache_dir.mkdir(parents=True, exist_ok=True)

def cache_readable_images(image_refs, limit=6):
    cached = []
    for idx, image_ref in enumerate(image_refs):
        if len(cached) >= limit:
            break

        try:
            if image_ref.startswith(('http://', 'https://')):
                response = requests.get(
                    image_ref,
                    timeout=30,
                    headers={'User-Agent': 'Mozilla/5.0'}
                )
                response.raise_for_status()
                img = Image.open(BytesIO(response.content)).convert('RGB')
                suffix = Path(urlparse(image_ref).path).suffix.lower()
                if suffix not in {'.jpg', '.jpeg', '.png', '.webp'}:
                    suffix = '.jpg'
                out_path = cache_dir / f'image_pool_{idx:02d}{suffix}'
                img.save(out_path)
                cached.append(str(out_path))
            else:
                img = Image.open(image_ref).convert('RGB')
                out_path = cache_dir / f'image_pool_{idx:02d}.jpg'
                img.save(out_path)
                cached.append(str(out_path))
        except Exception as exc:
            print(f'Skipping unreadable image: {image_ref} -> {exc}')

    return list(dict.fromkeys(cached))

sample_data = [{
    'image_url': image,
    'question': question,
    'answer': 'Painted Bunting'
}]

if not serpapi_api_key:
    raise RuntimeError(
        'Set SERPAPI_API_KEY before running the image_pool flow.'
    )

config = Config(
    raw_input='Identify the bird in the image.',
    sample_data=sample_data,
    input_fields=['question', 'image_url'],
    output_fields=['answer'],
    task_type='vqa',
    image_pool=[],
    image_pool_target_size=51,
    image_pool_sources=['serpapi'],
    auto_fetch_image_pool=True,
    serpapi_api_key=serpapi_api_key,
    model_provider=provider,
    model_name=optimizer_model,
    model_api_key=api_key,
    model_api_base=api_base,
    config_model_provider=provider,
    config_model_name=optimizer_model,
    config_model_api_key=api_key,
    config_model_api_base=api_base,
    backend='simple_meta_prompt',
    image_pool_use_lm_query=True,
)

query = config._build_image_pool_query()
print("query:", query)

raw_results = config._fetch_serpapi_image_candidates(
    query=query,
    limit=50,
)

print("Raw SerpApi results:", len(raw_results))
for i, url in enumerate(raw_results, 1):
    print(f"{i}. {url}")
print()

validated_results = config._resolve_existing_image_urls(
    raw_results,
    limit=50,
)

print("Validated image URLs:", len(validated_results))
for i, url in enumerate(validated_results, 1):
    print(f"{i}. {url}")
print()

print("Config image_pool:", len(config.image_pool))
for i, url in enumerate(config.image_pool, 1):
    print(f"{i}. {url}")

print("pool size after config:", len(config.image_pool))
for x in config.image_pool:
    print(x)

# 1.1: Config auto-populates candidate URLs from SerpApi.
fetched_candidates = [candidate for candidate in config.image_pool if candidate != image]
print('Fetched candidate URLs:', len(fetched_candidates))
for candidate in fetched_candidates:
    print(' -', candidate)
print()

# Convert fetched URLs into readable local files for 1.2.
config.image_pool = cache_readable_images(fetched_candidates, limit=50)
config.synthetic_data_size = len(config.image_pool)

if len(config.image_pool) < 3:
    raise RuntimeError(
        f'Only {len(config.image_pool)} readable images were cached from auto-fetched candidates. '
        'Try rerunning, raising image_pool_target_size, or using a different provider list.'
    )

optimizer = PromptOptimizer(config)
print('Configured image_pool size:', len(config.image_pool))
print('Configured image_pool files:')
for image_path in config.image_pool:
    print(' -', image_path)
print()
print('Configured task_type:', config.task_type)

query: painted bunting
Raw SerpApi results: 59
1. https://media.audubon.org/nas_birdapi_hero/h_a1_7443_5_painted-bunting_julie_torkomian_adult-male.jpg?width=1200&height=630&auto=webp&quality=90&fit=crop&enable=upscale
2. https://lookaside.fbsbx.com/lookaside/crawler/media/?media_id=10157071154673058
3. https://www.allaboutbirds.org/guide/assets/videoThumbs/616221708-480px.jpg
4. https://media.audubon.org/2023-06/Aud_APA-2018_Painted-Bunting_A1_7443-6_TS_Photo-Julie-Torkomian-1.jpg
5. https://inaturalist-open-data.s3.amazonaws.com/photos/293862224/large.jpeg
6. https://lookaside.instagram.com/seo/google_widget/crawler/?media_id=3613457674017319485
7. https://preview.redd.it/easily-one-of-the-most-colorful-birds-in-the-us-painted-v0-e2bgc4yxrf9c1.jpg?width=1080&crop=smart&auto=webp&s=6ba3cfb8e448258553352842a35548ce18fe246d
8. https://www.lyricbirdfood.com/media/1403/painted-bunting_davies-ian_ml31703301.jpg?crop=0,0.075181488080095435,0,0.039828632299152476&cropmode=percentage&width=70

In [7]:
first_sample, synthetic_samples = optimizer._prepare_sample_data()

print('Synthetic sample count:', len(synthetic_samples))
print('\nFirst sample:')
pprint(first_sample)

print('\nAll synthetic samples:')
for idx, sample in enumerate(synthetic_samples, start=1):
    print(f'--- sample {idx} ---')
    print('image_url:', sample.get('image_url', ''))
    print('question:', sample.get('question', ''))
    print('answer:', sample.get('answer', ''))
    print()


Synthetic sample count: 43

First sample:
{'answer': 'Red-orange',
 'image_url': 'https://media.audubon.org/nas_birdapi_hero/h_a1_7443_5_painted-bunting_julie_torkomian_adult-male.jpg',
 'question': "What is the primary color of the bird's chest?"}

All synthetic samples:
--- sample 1 ---
image_url: https://media.audubon.org/nas_birdapi_hero/h_a1_7443_5_painted-bunting_julie_torkomian_adult-male.jpg
question: What is the primary color of the bird's chest?
answer: Red-orange

--- sample 2 ---
image_url: https://i.ibb.co/hV4sDkC/image.jpg
question: What is the texture of the branch the bird is perched on?
answer: Rough and textured

--- sample 3 ---
image_url: https://i.ibb.co/bF0nC0N/image.jpg
question: What colors are visible on the bird's tail feathers?
answer: Blue and dark gray

--- sample 4 ---
image_url: https://images.unsplash.com/photo-1596707328639-c5c56d773410?crop=entropy&cs=tinysrgb&fit=max&fm=jpg&ixid=MnwxMTc3M3wwfDFyYW5kb218fHxiaXJkfHx8fHx8MTY3ODkzMDExMA&ixlib=rb-4.0.3&q=8

## Feed synthetic samples into the next stage

This cell takes the VQA samples synthesized from `image_pool` and turns them into `train_data` / `valid_data`, then prepares the DSPy examples the optimizer would consume next.

In [8]:
synthetic_dataset = optimizer.generate_synthetic_data()
split_idx = max(1, len(synthetic_dataset) // 2)

config.train_data = synthetic_dataset[:split_idx]
config.valid_data = synthetic_dataset[split_idx:] or synthetic_dataset[:1]

trainset, validset = optimizer._prepare_datasets()

print('Synthetic dataset size:', len(synthetic_dataset))
print('Train samples:', len(config.train_data))
print('Valid samples:', len(config.valid_data))
print('Prepared DSPy train examples:', len(trainset))
print('Prepared DSPy valid examples:', len(validset))

print('\nFirst prepared train example:')
print(trainset[0])


✅ Generated 43 multimodal synthetic samples


ValueError: Unrecognized file string: image_1_painted_bunting_closeup.jpg; If this file type should be supported, please open an issue.